# Entraînement AlphaZero Hex 11×11 — Google Colab GPU

**Instructions :**
1. Dans Colab : `Exécution > Modifier le type d'exécution > GPU (T4)`
2. Exécutez les cellules dans l'ordre
3. Les checkpoints sont sauvegardés dans Google Drive (`Mon Drive/hex_alphazero/`)

**Architecture réseau :** CNN ResNet (6 blocs, 128 canaux) — tête Politique + tête Valeur  
**Entraînement :** Self-play MCTS-PUCT + optimisation Adam

## 0. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/hex_alphazero'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'Dossier Drive : {DRIVE_DIR}')

## 1. Vérifier le GPU

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Écrire les fichiers source

In [ ]:
import os
os.makedirs('/content/alphazero', exist_ok=True)
os.chdir('/content/alphazero')
print('Répertoire de travail :', os.getcwd())

In [ ]:
%%writefile config.py
# config.py — Hyperparamètres centralisés pour AlphaZero Hex 11×11

# ─── Plateau ──────────────────────────────────────────────────────────────────
BOARD_SIZE = 11
NUM_CELLS  = BOARD_SIZE * BOARD_SIZE  # 121

# ─── Réseau ───────────────────────────────────────────────────────────────────
NUM_CHANNELS    = 128   # filtres par couche convolutive
NUM_RES_BLOCKS  = 6     # blocs résiduels
INPUT_CHANNELS  = 3     # (Blue, Red, joueur_courant)

# ─── MCTS ─────────────────────────────────────────────────────────────────────
MCTS_SIMULATIONS    = 400   # simulations par coup (entraînement)
MCTS_SIMULATIONS_EVAL = 200 # simulations par coup (évaluation)
C_PUCT              = 1.0   # constante d'exploration UCB-PUCT
DIRICHLET_ALPHA     = 0.3   # bruit Dirichlet à la racine
DIRICHLET_EPS       = 0.25  # poids du bruit Dirichlet
TEMPERATURE_MOVES   = 15    # coups joués avec τ=1 avant de passer à τ→0

# ─── Self-play ────────────────────────────────────────────────────────────────
GAMES_PER_ITER      = 100   # parties par itération de self-play
REPLAY_BUFFER_SIZE  = 50_000  # taille du buffer circulaire (positions)

# ─── Entraînement ─────────────────────────────────────────────────────────────
BATCH_SIZE          = 512
LEARNING_RATE       = 1e-3
LR_SCHEDULER        = "cosine"
WEIGHT_DECAY        = 1e-4
TRAIN_STEPS         = 1000   # steps par itération

# ─── Évaluation ───────────────────────────────────────────────────────────────
EVAL_GAMES          = 40     # parties d'évaluation (20 par couleur)
WIN_RATE_THRESHOLD  = 0.55   # seuil pour accepter le nouveau modèle

# ─── Checkpoints ──────────────────────────────────────────────────────────────
CHECKPOINT_DIR  = '/content/drive/MyDrive/hex_alphazero/checkpoints'
BEST_MODEL_FILE = '/content/drive/MyDrive/hex_alphazero/checkpoints/best_model.pt'

In [ ]:
%%writefile hex_env.py
# hex_env.py — Moteur Hex 11×11 en Python/numpy
# Blue (O) connecte Nord (ligne 0) → Sud (ligne 10)
# Red  (@) connecte Ouest (col 0)  → Est  (col 10)

import numpy as np
from collections import deque
from config import BOARD_SIZE, NUM_CELLS

_HEX_NEIGHBORS = [(-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0)]


class HexEnv:
    def __init__(self):
        self.blue  = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=bool)
        self.red   = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=bool)
        self.blue_to_play = True
        self._winner = None

    @classmethod
    def from_string(cls, board_str: str, player_char: str) -> "HexEnv":
        env = cls()
        for i, ch in enumerate(board_str[:NUM_CELLS]):
            r, c = divmod(i, BOARD_SIZE)
            if ch == 'O':
                env.blue[r, c] = True
            elif ch == '@':
                env.red[r, c] = True
        env.blue_to_play = (player_char == 'O')
        return env

    def get_legal_moves(self) -> np.ndarray:
        occupied = self.blue | self.red
        return np.where(~occupied.ravel())[0]

    def legal_mask(self) -> np.ndarray:
        occupied = self.blue | self.red
        return ~occupied.ravel()

    def apply_move(self, pos: int) -> None:
        r, c = divmod(pos, BOARD_SIZE)
        if self.blue_to_play:
            self.blue[r, c] = True
        else:
            self.red[r, c] = True
        self._winner = None
        self.blue_to_play = not self.blue_to_play

    def _blue_wins(self) -> bool:
        visited = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=bool)
        queue = deque()
        for c in range(BOARD_SIZE):
            if self.blue[0, c] and not visited[0, c]:
                visited[0, c] = True
                queue.append((0, c))
        while queue:
            r, c = queue.popleft()
            if r == BOARD_SIZE - 1:
                return True
            for dr, dc in _HEX_NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < BOARD_SIZE and 0 <= nc < BOARD_SIZE:
                    if self.blue[nr, nc] and not visited[nr, nc]:
                        visited[nr, nc] = True
                        queue.append((nr, nc))
        return False

    def _red_wins(self) -> bool:
        visited = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=bool)
        queue = deque()
        for r in range(BOARD_SIZE):
            if self.red[r, 0] and not visited[r, 0]:
                visited[r, 0] = True
                queue.append((r, 0))
        while queue:
            r, c = queue.popleft()
            if c == BOARD_SIZE - 1:
                return True
            for dr, dc in _HEX_NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < BOARD_SIZE and 0 <= nc < BOARD_SIZE:
                    if self.red[nr, nc] and not visited[nr, nc]:
                        visited[nr, nc] = True
                        queue.append((nr, nc))
        return False

    def is_terminal(self) -> bool:
        if self._winner is not None:
            return True
        if self._blue_wins():
            self._winner = 'blue'
            return True
        if self._red_wins():
            self._winner = 'red'
            return True
        return False

    def winner(self) -> str | None:
        self.is_terminal()
        return self._winner

    def get_state_tensor(self) -> np.ndarray:
        tensor = np.zeros((3, BOARD_SIZE, BOARD_SIZE), dtype=np.float32)
        tensor[0] = self.blue.astype(np.float32)
        tensor[1] = self.red.astype(np.float32)
        tensor[2] = 1.0 if self.blue_to_play else 0.0
        return tensor

    def copy(self) -> "HexEnv":
        env = HexEnv()
        env.blue = self.blue.copy()
        env.red  = self.red.copy()
        env.blue_to_play = self.blue_to_play
        env._winner = self._winner
        return env

    def mirror(self) -> "HexEnv":
        env = HexEnv()
        env.blue = self.red.T.copy()
        env.red  = self.blue.T.copy()
        env.blue_to_play = not self.blue_to_play
        env._winner = None
        return env

    def to_string(self) -> str:
        chars = []
        for r in range(BOARD_SIZE):
            for c in range(BOARD_SIZE):
                if self.blue[r, c]:
                    chars.append('O')
                elif self.red[r, c]:
                    chars.append('@')
                else:
                    chars.append('.')
        return ''.join(chars)

    def pos_to_str(self, pos: int) -> str:
        r, c = divmod(pos, BOARD_SIZE)
        return f"{chr(ord('A') + c)}{r + 1}"

    @staticmethod
    def str_to_pos(s: str) -> int:
        col = ord(s[0].upper()) - ord('A')
        row = int(s[1:]) - 1
        return row * BOARD_SIZE + col

    def __str__(self) -> str:
        lines = []
        header = "  " + " ".join(chr(ord('A') + c) for c in range(BOARD_SIZE))
        lines.append(header)
        for r in range(BOARD_SIZE):
            prefix = " " * r
            row_str = " ".join(
                'O' if self.blue[r, c] else ('@' if self.red[r, c] else '.')
                for c in range(BOARD_SIZE)
            )
            lines.append(f"{prefix}{r+1:2d} {row_str}")
        return "\n".join(lines)

In [ ]:
%%writefile network.py
# network.py — Réseau de neurones CNN pour AlphaZero Hex 11×11

import torch
import torch.nn as nn
import torch.nn.functional as F
from config import NUM_CELLS, NUM_CHANNELS, NUM_RES_BLOCKS, INPUT_CHANNELS


class ResBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class HexNet(nn.Module):
    def __init__(self, in_channels=INPUT_CHANNELS, channels=NUM_CHANNELS, num_blocks=NUM_RES_BLOCKS):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.res_blocks = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.policy_conv = nn.Conv2d(channels, 2, 1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * NUM_CELLS, NUM_CELLS)
        self.value_conv  = nn.Conv2d(channels, 1, 1, bias=False)
        self.value_bn    = nn.BatchNorm2d(1)
        self.value_fc1   = nn.Linear(NUM_CELLS, 256)
        self.value_fc2   = nn.Linear(256, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.res_blocks(x)
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        p = self.policy_fc(p)
        log_p = F.log_softmax(p, dim=1)
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))
        return log_p, v

    def predict(self, state_tensor, legal_mask, device):
        import numpy as np
        self.eval()
        with torch.no_grad():
            t = torch.from_numpy(state_tensor).unsqueeze(0).to(device)
            log_p, v = self(t)
            log_p = log_p.squeeze(0).cpu().numpy()
            value  = v.squeeze().item()
        policy = np.exp(log_p)
        policy = policy * legal_mask.astype(np.float32)
        s = policy.sum()
        if s > 1e-8:
            policy /= s
        else:
            policy = legal_mask.astype(np.float32)
            policy /= policy.sum()
        return policy, value

In [ ]:
%%writefile mcts_az.py
# mcts_az.py — MCTS guidé par réseau de neurones (UCB-PUCT style AlphaZero)

import math
import numpy as np
import torch

from hex_env import HexEnv
from config import C_PUCT, MCTS_SIMULATIONS, DIRICHLET_ALPHA, DIRICHLET_EPS, TEMPERATURE_MOVES, NUM_CELLS


class MCTSNode:
    __slots__ = ('env', 'parent', 'move', 'children', 'N', 'W', 'Q', 'P', 'is_expanded', 'is_terminal')

    def __init__(self, env, parent=None, move=-1, prior=0.0):
        self.env = env
        self.parent = parent
        self.move = move
        self.children = {}
        self.N = 0
        self.W = 0.0
        self.Q = 0.0
        self.P = prior
        self.is_expanded = False
        self.is_terminal = False


class MCTSAgent:
    def __init__(self, net, device=None, sims=MCTS_SIMULATIONS, c_puct=C_PUCT, add_dirichlet=True):
        self.net = net
        self.device = device or torch.device('cpu')
        self.sims = sims
        self.c_puct = c_puct
        self.add_dirichlet = add_dirichlet

    def _evaluate(self, env):
        legal_mask = env.legal_mask()
        if self.net is None:
            n_legal = legal_mask.sum()
            policy = legal_mask.astype(np.float32) / max(n_legal, 1)
            return policy, 0.0
        state = env.get_state_tensor()
        policy, value = self.net.predict(state, legal_mask, self.device)
        return policy, value

    def _select_child(self, node):
        sqrt_N = math.sqrt(max(node.N, 1))
        best_score = -float('inf')
        best_move = -1
        for move, child in node.children.items():
            q = -child.Q if child.N > 0 else 0.0
            u = self.c_puct * child.P * sqrt_N / (1 + child.N)
            score = q + u
            if score > best_score:
                best_score = score
                best_move = move
        return best_move

    def _expand(self, node):
        if node.env.is_terminal():
            node.is_terminal = True
            return -1.0
        policy, value = self._evaluate(node.env)
        node.is_expanded = True
        for move in range(NUM_CELLS):
            if policy[move] > 0:
                child_env = node.env.copy()
                child_env.apply_move(move)
                node.children[move] = MCTSNode(env=child_env, parent=node, move=move, prior=float(policy[move]))
        return value

    def _backprop(self, node, value):
        cur = node
        v = value
        while cur is not None:
            cur.N += 1
            cur.W += v
            cur.Q = cur.W / cur.N
            v = -v
            cur = cur.parent

    def _simulate(self, root):
        node = root
        while node.is_expanded and not node.is_terminal:
            move = self._select_child(node)
            if move == -1:
                break
            node = node.children[move]
        value = self._expand(node)
        self._backprop(node, value)

    def get_policy(self, env, move_count=0, return_root=False):
        root_env = env.copy()
        root = MCTSNode(root_env)
        self._expand(root)
        if self.add_dirichlet and root.children:
            moves = list(root.children.keys())
            noise = np.random.dirichlet([DIRICHLET_ALPHA] * len(moves))
            for m, n in zip(moves, noise):
                child = root.children[m]
                child.P = (1 - DIRICHLET_EPS) * child.P + DIRICHLET_EPS * n
        for _ in range(self.sims - 1):
            self._simulate(root)
        pi = np.zeros(NUM_CELLS, dtype=np.float32)
        for move, child in root.children.items():
            pi[move] = child.N
        if move_count < TEMPERATURE_MOVES:
            s = pi.sum()
            if s > 0:
                pi /= s
        else:
            best = pi.argmax()
            pi[:] = 0.0
            pi[best] = 1.0
        if return_root:
            return pi, root
        return pi

    def select_move(self, env, move_count=0):
        pi = self.get_policy(env, move_count=move_count)
        if move_count < TEMPERATURE_MOVES:
            return int(np.random.choice(NUM_CELLS, p=pi))
        else:
            return int(pi.argmax())

In [ ]:
%%writefile self_play.py
# self_play.py — Génération de parties en self-play pour AlphaZero Hex

import numpy as np
from collections import deque

from hex_env import HexEnv
from mcts_az import MCTSAgent
from config import REPLAY_BUFFER_SIZE, GAMES_PER_ITER, MCTS_SIMULATIONS, NUM_CELLS


class ReplayBuffer:
    def __init__(self, max_size=REPLAY_BUFFER_SIZE):
        self.max_size = max_size
        self._states   = deque(maxlen=max_size)
        self._policies = deque(maxlen=max_size)
        self._values   = deque(maxlen=max_size)

    def add(self, state, policy, value):
        self._states.append(state)
        self._policies.append(policy)
        self._values.append(value)

    def __len__(self):
        return len(self._states)

    def sample(self, batch_size):
        idx = np.random.choice(len(self._states), size=batch_size, replace=False)
        states   = np.stack([self._states[i]   for i in idx])
        policies = np.stack([self._policies[i] for i in idx])
        values   = np.array([self._values[i]   for i in idx], dtype=np.float32)
        return states, policies, values


def _augment(state, policy, value):
    state_aug = np.zeros_like(state)
    state_aug[0] = state[1].T
    state_aug[1] = state[0].T
    state_aug[2] = 1.0 - state[2]
    policy_aug = np.zeros(NUM_CELLS, dtype=np.float32)
    for i in range(NUM_CELLS):
        r, c = divmod(i, 11)
        j = c * 11 + r
        policy_aug[j] = policy[i]
    return state_aug, policy_aug, -value


def play_one_game(agent, buffer, augment=True, verbose=False):
    env = HexEnv()
    history = []
    move_count = 0
    while not env.is_terminal():
        pi = agent.get_policy(env, move_count=move_count)
        state = env.get_state_tensor()
        history.append((state.copy(), pi.copy(), env.blue_to_play))
        move = int(np.random.choice(NUM_CELLS, p=pi))
        if verbose:
            r, c = divmod(move, 11)
            print(f"  Coup {move_count+1}: {chr(ord('A')+c)}{r+1} ({'Blue' if env.blue_to_play else 'Red'})")
        env.apply_move(move)
        move_count += 1
    gagnant = env.winner()
    for state, pi, was_blue_to_play in history:
        if gagnant == 'blue':
            z = 1.0 if was_blue_to_play else -1.0
        else:
            z = -1.0 if was_blue_to_play else 1.0
        buffer.add(state, pi, float(z))
        if augment:
            s_aug, p_aug, z_aug = _augment(state, pi, float(z))
            buffer.add(s_aug, p_aug, z_aug)
    if verbose:
        print(f"  → Vainqueur : {gagnant} en {move_count} coups")
    return gagnant


def run_self_play(agent, buffer, num_games=GAMES_PER_ITER, verbose=False):
    stats = {'blue_wins': 0, 'red_wins': 0, 'total': num_games}
    for i in range(num_games):
        winner = play_one_game(agent, buffer, augment=True, verbose=verbose)
        if winner == 'blue':
            stats['blue_wins'] += 1
        else:
            stats['red_wins'] += 1
        if verbose or (i + 1) % 10 == 0:
            print(f"  Partie {i+1}/{num_games} terminée | buffer={len(buffer)}")
    return stats

In [ ]:
%%writefile evaluate.py
# evaluate.py — Comparaison de deux modèles HexNet par tournoi interne

import numpy as np
import torch

from hex_env import HexEnv
from mcts_az import MCTSAgent
from network import HexNet
from config import MCTS_SIMULATIONS_EVAL, EVAL_GAMES, NUM_CELLS


def _play_game(agent_blue, agent_red):
    env = HexEnv()
    move_count = 0
    while not env.is_terminal():
        if env.blue_to_play:
            pi = agent_blue.get_policy(env, move_count=999)
        else:
            pi = agent_red.get_policy(env, move_count=999)
        move = int(pi.argmax())
        env.apply_move(move)
        move_count += 1
    return env.winner()


def evaluate_models(new_net, best_net, device, num_games=EVAL_GAMES, sims=MCTS_SIMULATIONS_EVAL):
    new_net.eval()
    best_net.eval()
    agent_new  = MCTSAgent(new_net,  device=device, sims=sims, add_dirichlet=False)
    agent_best = MCTSAgent(best_net, device=device, sims=sims, add_dirichlet=False)
    wins = 0
    half = num_games // 2
    for i in range(half):
        winner = _play_game(agent_new, agent_best)
        if winner == 'blue':
            wins += 1
        if (i + 1) % 5 == 0:
            print(f"    Éval Blue {i+1}/{half} — new wins={wins}")
    for i in range(num_games - half):
        winner = _play_game(agent_best, agent_new)
        if winner == 'red':
            wins += 1
        if (i + 1) % 5 == 0:
            print(f"    Éval Red {i+1}/{num_games-half} — new wins={wins}")
    return wins / num_games


def evaluate_vs_random(net, device, num_games=20, sims=50):
    net.eval()
    agent = MCTSAgent(net, device=device, sims=sims, add_dirichlet=False)
    wins = 0
    for i in range(num_games):
        env = HexEnv()
        net_is_blue = (i % 2 == 0)
        move_count = 0
        while not env.is_terminal():
            if env.blue_to_play == net_is_blue:
                pi = agent.get_policy(env, move_count=999)
                move = int(pi.argmax())
            else:
                legal = env.get_legal_moves()
                move = int(np.random.choice(legal))
            env.apply_move(move)
            move_count += 1
        winner = env.winner()
        if (net_is_blue and winner == 'blue') or (not net_is_blue and winner == 'red'):
            wins += 1
    return wins / num_games

In [ ]:
%%writefile trainer.py
# trainer.py — Boucle d'entraînement AlphaZero pour Hex 11×11

import argparse
import os
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F

from network import HexNet
from mcts_az import MCTSAgent
from self_play import ReplayBuffer, run_self_play
from evaluate import evaluate_models
from config import (
    BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, TRAIN_STEPS,
    GAMES_PER_ITER, MCTS_SIMULATIONS, CHECKPOINT_DIR, BEST_MODEL_FILE,
    REPLAY_BUFFER_SIZE, WIN_RATE_THRESHOLD, EVAL_GAMES,
)


def train_step(net, optimizer, states, policies, values, device):
    net.train()
    states_t   = torch.from_numpy(states).to(device)
    policies_t = torch.from_numpy(policies).to(device)
    values_t   = torch.from_numpy(values).unsqueeze(1).to(device)
    log_p, v = net(states_t)
    loss_policy = -(policies_t * log_p).sum(dim=1).mean()
    loss_value  = F.mse_loss(v, values_t)
    loss = loss_policy + loss_value
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item(), loss_value.item(), loss_policy.item()


def train_epoch(net, optimizer, scheduler, buffer, steps, batch_size, device):
    losses, losses_v, losses_p = [], [], []
    for _ in range(steps):
        if len(buffer) < batch_size:
            break
        states, policies, values = buffer.sample(batch_size)
        l, lv, lp = train_step(net, optimizer, states, policies, values, device)
        losses.append(l)
        losses_v.append(lv)
        losses_p.append(lp)
    if scheduler is not None:
        scheduler.step()
    return {
        'loss':        float(np.mean(losses))   if losses   else 0.0,
        'loss_value':  float(np.mean(losses_v)) if losses_v else 0.0,
        'loss_policy': float(np.mean(losses_p)) if losses_p else 0.0,
    }


def save_checkpoint(net, path):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    torch.save(net.state_dict(), path)
    print(f'  Checkpoint sauvegardé : {path}')


def load_checkpoint(net, path, device):
    if os.path.isfile(path):
        net.load_state_dict(torch.load(path, map_location=device))
        print(f'  Checkpoint chargé : {path}')
        return True
    return False


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--iterations',  type=int, default=20)
    parser.add_argument('--games',       type=int, default=GAMES_PER_ITER)
    parser.add_argument('--simulations', type=int, default=MCTS_SIMULATIONS)
    parser.add_argument('--steps',       type=int, default=TRAIN_STEPS)
    parser.add_argument('--batch',       type=int, default=BATCH_SIZE)
    parser.add_argument('--device',      type=str, default='auto')
    parser.add_argument('--eval-games',  type=int, default=EVAL_GAMES)
    parser.add_argument('--no-eval',     action='store_true')
    args = parser.parse_args()

    if args.device == 'auto':
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    else:
        device = torch.device(args.device)
    print(f'Device : {device}')

    net = HexNet().to(device)
    if not load_checkpoint(net, BEST_MODEL_FILE, device):
        print('  Nouveau modèle initialisé.')

    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.iterations)
    buffer = ReplayBuffer(REPLAY_BUFFER_SIZE)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    for iteration in range(1, args.iterations + 1):
        print(f"\n{'='*60}")
        print(f'Itération {iteration}/{args.iterations}')
        t0 = time.time()

        print(f'\n[1/3] Self-play ({args.games} parties, {args.simulations} sims)...')
        agent = MCTSAgent(net, device=device, sims=args.simulations, add_dirichlet=True)
        stats = run_self_play(agent, buffer, num_games=args.games)
        print(f'  Blue : {stats["blue_wins"]} | Red : {stats["red_wins"]} | Buffer : {len(buffer)}')

        if len(buffer) < args.batch:
            print(f'  Buffer trop petit ({len(buffer)} < {args.batch}), prochaine itération.')
            continue

        print(f'\n[2/3] Entraînement ({args.steps} steps, batch={args.batch})...')
        metrics = train_epoch(net, optimizer, scheduler, buffer, args.steps, args.batch, device)
        print(f'  Loss={metrics["loss"]:.4f}  valeur={metrics["loss_value"]:.4f}  politique={metrics["loss_policy"]:.4f}')

        iter_path = os.path.join(CHECKPOINT_DIR, f'model_iter_{iteration:04d}.pt')
        save_checkpoint(net, iter_path)

        if not args.no_eval:
            print(f'\n[3/3] Évaluation ({args.eval_games} parties)...')
            best_net = HexNet().to(device)
            if not load_checkpoint(best_net, BEST_MODEL_FILE, device):
                save_checkpoint(net, BEST_MODEL_FILE)
            else:
                win_rate = evaluate_models(net, best_net, device, num_games=args.eval_games)
                print(f'  Win rate : {win_rate:.1%} (seuil={WIN_RATE_THRESHOLD:.0%})')
                if win_rate >= WIN_RATE_THRESHOLD:
                    print('  ✓ Nouveau modèle accepté !')
                    save_checkpoint(net, BEST_MODEL_FILE)
                else:
                    print('  ✗ Ancien modèle conservé.')
                    load_checkpoint(net, BEST_MODEL_FILE, device)

        print(f'\n  Durée itération : {time.time()-t0:.1f}s')

    print('\nEntraînement terminé.')
    print(f'Meilleur modèle : {BEST_MODEL_FILE}')


if __name__ == '__main__':
    main()

## 3. Lancer l'entraînement

**Paramètres recommandés pour Colab T4 :**
- `--iterations 50` : 50 cycles self-play → entraînement
- `--games 50` : 50 parties par itération (réduit pour Colab)
- `--simulations 200` : 200 sims MCTS par coup
- `--steps 500` : 500 steps d'entraînement par itération
- `--no-eval` : désactive l'évaluation pour aller plus vite

> **Durée estimée :** ~15-30 min par itération. Utilisez `--no-eval` pour accélérer.

In [ ]:
# Lancer l'entraînement (ajustez les paramètres selon vos besoins)
!python trainer.py \
    --iterations 50 \
    --games 50 \
    --simulations 200 \
    --steps 500 \
    --batch 256 \
    --device cuda \
    --no-eval

## 4. Reprendre un entraînement interrompu

Si la session Colab s'interrompt, le meilleur modèle est dans Google Drive.  
Ré-exécutez les cellules 0→2, puis cette cellule pour reprendre.

In [ ]:
# Reprendre depuis le dernier checkpoint
!python trainer.py \
    --iterations 50 \
    --games 50 \
    --simulations 200 \
    --steps 500 \
    --batch 256 \
    --device cuda \
    --no-eval
# Le script détecte automatiquement best_model.pt dans Drive et repart de là

## 5. Évaluer le modèle entraîné vs joueur aléatoire

In [ ]:
import sys
sys.path.insert(0, '/content/alphazero')

import torch
from network import HexNet
from evaluate import evaluate_vs_random
from config import BEST_MODEL_FILE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = HexNet().to(device)
net.load_state_dict(torch.load(BEST_MODEL_FILE, map_location=device))
print(f'Modèle chargé : {BEST_MODEL_FILE}')

wr = evaluate_vs_random(net, device, num_games=20, sims=100)
print(f'\nWin rate vs aléatoire : {wr:.1%}')

## 6. Télécharger le meilleur modèle

In [ ]:
from google.colab import files
files.download(BEST_MODEL_FILE)